In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 16g pyspark-shell"

import json
import math
from collections import defaultdict, namedtuple
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.statcounter import StatCounter
from shapely.geometry import Point, shape

In [2]:
spark = SparkSession.builder.appName("RunTaxiTrips").master("local[*]").getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Cores in use:", sc.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/28 18:06:09 WARN Utils: Your hostname, fedora, resolves to a loopback address: 127.0.0.1; using 192.168.0.108 instead (on interface wlo1)
26/05/28 18:06:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 18:06:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Cores in use: 8


In [3]:
# ---- Parsing ----

TaxiTrip = namedtuple(
    "TaxiTrip", ["pickupTime", "dropoffTime", "pickupLoc", "dropoffLoc"]
)


def parse(line: str):
    fields = line.split(",")
    license = fields[1]
    pickupTime = datetime.strptime(fields[5], "%Y-%m-%d %H:%M:%S")
    dropoffTime = datetime.strptime(fields[6], "%Y-%m-%d %H:%M:%S")
    pickupLoc = (float(fields[10]), float(fields[11]))  # (lon, lat)
    dropoffLoc = (float(fields[12]), float(fields[13]))  # (lon, lat)
    return (license, TaxiTrip(pickupTime, dropoffTime, pickupLoc, dropoffLoc))


def safe(f):
    def wrapper(s):
        try:
            return f(s)
        except Exception as e:
            return (s, e)

    return wrapper


def get_duration_seconds(trip):
    return (trip.dropoffTime - trip.pickupTime).total_seconds()


def get_duration_hours(trip):
    return int(get_duration_seconds(trip) / 3600)


# ---- Distance & normalisation ----


def haversine_km(loc1, loc2):
    """Great-circle distance in km between two (lon, lat) points."""
    lon1, lat1 = loc1
    lon2, lat2 = loc2
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))


def normalized_duration(trip):
    """Duration in seconds divided by straight-line distance in km. None if distance is zero."""
    dist = haversine_km(trip.pickupLoc, trip.dropoffLoc)
    if dist == 0.0:
        return None
    return get_duration_seconds(trip) / dist


# ---- Compact cached record: borough + metrics pre-computed once ----

TripRecord = namedtuple(
    "TripRecord",
    [
        "license",
        "pickup_ts",  # float: POSIX timestamp — used for sort and gap detection
        "dropoff_ts",  # float
        "pickup_hour",  # int 0-23
        "pickup_boro",  # str or None
        "dropoff_boro",  # str or None
        "dur_sec",  # float
        "norm_dur",  # float or None (dur_sec / haversine_km)
    ],
)

In [4]:
# ---- Borough polygon lookup — must be defined before the load pipeline ----

with open("../../data/problem_2/nyc-borough-boundaries-polygon.geojson.json", "r") as f:
    geojson = json.load(f)

bFeatures = sc.broadcast(geojson["features"])


def borough_for_loc(lon, lat):
    p = Point(lon, lat)
    for feature in bFeatures.value:
        if p.within(shape(feature["geometry"])):
            return feature["properties"]["borough"]
    return None


def pickup_borough(trip):
    return borough_for_loc(*trip.pickupLoc)


def dropoff_borough(trip):
    return borough_for_loc(*trip.dropoffLoc)

In [5]:
taxiRaw = spark.read.text("../../data/problem_2/trip_data-part*.csv")

taxiDone = (
    taxiRaw.rdd.map(lambda row: safe(parse)(row.value))
    .filter(lambda x: not isinstance(x[1], Exception))
    .filter(lambda x: 0 <= get_duration_hours(x[1]) < 3)
    .filter(lambda x: x[1].pickupLoc != (0.0, 0.0) and x[1].dropoffLoc != (0.0, 0.0))
    .map(
        lambda x: TripRecord(
            license=x[0],
            pickup_ts=x[1].pickupTime.timestamp(),
            dropoff_ts=x[1].dropoffTime.timestamp(),
            pickup_hour=x[1].pickupTime.hour,
            pickup_boro=pickup_borough(x[1]),
            dropoff_boro=dropoff_borough(x[1]),
            dur_sec=get_duration_seconds(x[1]),
            norm_dur=normalized_duration(x[1]),
        )
    )
)
taxiDone.cache()

print("Total clean trips:", taxiDone.count())

ERROR:root:KeyboardInterrupt while sending command.                (0 + 8) / 20]
Traceback (most recent call last):
  File "/home/isaac/Documents/BDA/bda-exercises-4-group-1/solutions/problem_2/env/lib64/python3.14/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/isaac/Documents/BDA/bda-exercises-4-group-1/solutions/problem_2/env/lib64/python3.14/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib64/python3.14/socket.py", line 725, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt
26/05/28 20:48:29 ERROR Executor: Exception in task 4.0 in stage 0.0 (TID 4)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/isaac/Apps/spark-4.0.2-bin-hadoop3/python/lib/pyspark.zip/pyspark/worker.py", line 2

KeyboardInterrupt: 

## (a) Trips that started AND ended in the same borough

In [ ]:
same_borough_stats = (
    taxiDone.filter(
        lambda r: r.pickup_boro is not None
        and r.dropoff_boro is not None
        and r.pickup_boro == r.dropoff_boro
    )
    .map(lambda r: (r.pickup_boro, r.dur_sec))
    .mapValues(lambda d: StatCounter().merge(d))
    .reduceByKey(lambda a, b: a.mergeStats(b))
    .collect()
)

print("Trips starting and ending in the SAME borough (duration in seconds):")
for borough, stats in sorted(same_borough_stats, key=lambda x: x[0] or ""):
    print(
        f"  {str(borough):15s}  count={int(stats.count()):>8,}  sum={stats.sum():>16,.0f}  mean={stats.mean():>8.1f}"
    )

## (b) Trips that started and ended in different boroughs

In [ ]:
cross_borough_stats = (
    taxiDone.filter(
        lambda r: r.pickup_boro is not None
        and r.dropoff_boro is not None
        and r.pickup_boro != r.dropoff_boro
    )
    .map(lambda r: ((r.pickup_boro, r.dropoff_boro), r.dur_sec))
    .mapValues(lambda d: StatCounter().merge(d))
    .reduceByKey(lambda a, b: a.mergeStats(b))
    .collect()
)

print("Trips starting and ending in DIFFERENT boroughs (duration in seconds):")
for (pb, db), stats in sorted(cross_borough_stats, key=lambda x: x[0]):
    print(
        f"  {pb:15s} -> {db:15s}  count={int(stats.count()):>7,}  sum={stats.sum():>14,.0f}  mean={stats.mean():>8.1f}"
    )

## Sessionization

Groups consecutive trips by the same driver into sessions, splitting when the gap between pickup times exceeds 4 hours.

In [ ]:
def secondaryKey(r):
    return r.pickup_ts


def split(r1, r2):
    return (r2.pickup_ts - r1.pickup_ts) / 3600 >= 4


def groupSorted(it, splitFunc):
    """
    Receives an iterator of ((license, ts), TripRecord) already sorted by (license, ts)
    within the partition. Emits (license, [TripRecord, ...]) sessions, starting a new
    session on driver change OR when splitFunc(prev, cur) is True.
    """
    current_license = None
    current_session = []
    for key, value in it:
        license = key[0]
        if current_license is None:
            current_license = license
            current_session = [value]
        elif license != current_license or splitFunc(current_session[-1], value):
            yield (current_license, current_session)
            current_license = license
            current_session = [value]
        else:
            current_session.append(value)
    if current_session:
        yield (current_license, current_session)


def groupByKeyAndSortValues(rdd, secondaryKeyFunc, splitFunc, numPartitions):
    presess = rdd.map(lambda x: ((x[0], secondaryKeyFunc(x[1])), x[1]))
    return (
        presess.partitionBy(numPartitions)
        .sortByKey()
        .mapPartitions(lambda it: groupSorted(it, splitFunc))
    )


# Use 100 partitions for the full dataset (vs 30 for a day sample)
sessions = groupByKeyAndSortValues(
    taxiDone.map(lambda r: (r.license, r)), secondaryKey, split, 100
)
sessions.cache()

print("Total sessions:", sessions.count())

## (c) Average wait-time between consecutive trips per borough AND hour-of-day

In [ ]:
def boroughDurationHour(r1, r2):
    """Wait time between two consecutive trips in the same session."""
    d = r2.pickup_ts - r1.dropoff_ts  # seconds between drop-off and next pick-up
    return ((r1.pickup_boro, r1.pickup_hour), d)


borough_hour_stats = (
    sessions.values()
    .flatMap(
        lambda trips: (
            boroughDurationHour(trips[i], trips[i + 1]) for i in range(len(trips) - 1)
        )
    )
    .filter(lambda x: x[1] >= 0)
    .mapValues(lambda d: StatCounter().merge(d))
    .reduceByKey(lambda a, b: a.mergeStats(b))
    .collect()
)

print("Average wait-time between consecutive trips (seconds) per borough and hour:")
for (borough, hour), stats in sorted(
    borough_hour_stats, key=lambda x: (x[0][0] or "", x[0][1])
):
    print(
        f"  {str(borough):15s}  hour={hour:02d}  count={int(stats.count()):>6,}  mean={stats.mean():>8.1f}s"
    )

## (d) Outlier detection — trips above the 95th percentile of normalised duration per borough

In [ ]:
# norm_dur is pre-computed in TripRecord — no repeated distance calculation needed
norm_rdd = taxiDone.filter(
    lambda r: r.pickup_boro is not None and r.norm_dur is not None
).map(lambda r: (r.pickup_boro, r.norm_dur))
norm_rdd.cache()

borough_thresholds = (
    norm_rdd.groupByKey()
    .mapValues(lambda vals: sorted(vals))
    .mapValues(lambda sv: sv[int(0.95 * len(sv))])
    .collectAsMap()
)

print("95th-percentile normalised duration (s/km) per borough:")
for b, thr in sorted(borough_thresholds.items(), key=lambda x: x[0] or ""):
    print(f"  {str(b):15s}  threshold = {thr:.2f} s/km")

thresholds_bc = sc.broadcast(borough_thresholds)

outlier_counts = norm_rdd.filter(
    lambda x: x[1] > thresholds_bc.value.get(x[0], float("inf"))
).countByKey()

print("\nOutlier trip counts per borough:")
for b, cnt in sorted(outlier_counts.items(), key=lambda x: x[0] or ""):
    print(f"  {str(b):15s}  outliers = {cnt:,}")

norm_rdd.unpersist()

## (e) Rush-hour detection — average normalised duration per hour-of-day (descending)

In [ ]:
# norm_dur and pickup_hour are pre-computed in TripRecord
rush_hour_stats = (
    taxiDone.filter(lambda r: r.norm_dur is not None)
    .map(lambda r: (r.pickup_hour, r.norm_dur))
    .mapValues(lambda d: StatCounter().merge(d))
    .reduceByKey(lambda a, b: a.mergeStats(b))
    .map(lambda x: (x[0], x[1].mean()))
    .collect()
)

rush_hour_stats.sort(key=lambda x: -x[1])

print("Average normalised trip duration (s/km) per hour-of-day, descending:")
for hour, mean_norm in rush_hour_stats:
    print(f"  hour={hour:02d}  avg = {mean_norm:.2f} s/km")